In [ ]:
import zipfile
import os

zip_path = "/content/Datasets.zip"
extract_path = "/content/datasets"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Carpeta creada:", os.path.exists(extract_path))
print("Archivos extraídos:", os.listdir(extract_path))

Carpeta creada: True
Archivos extraídos: ['vendors.csv', 'transaction_items.csv', 'store_promotions.csv', 'stores.csv', 'transactions.csv', 'products.csv']


In [ ]:
!pip -q install duckdb

import duckdb
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW transactions AS
SELECT * FROM read_csv_auto('/content/datasets/transactions.csv', HEADER=TRUE);

CREATE OR REPLACE VIEW transaction_items AS
SELECT * FROM read_csv_auto('/content/datasets/transaction_items.csv', HEADER=TRUE);

CREATE OR REPLACE VIEW stores AS
SELECT * FROM read_csv_auto('/content/datasets/stores.csv', HEADER=TRUE);

CREATE OR REPLACE VIEW products AS
SELECT * FROM read_csv_auto('/content/datasets/products.csv', HEADER=TRUE);

CREATE OR REPLACE VIEW vendors AS
SELECT * FROM read_csv_auto('/content/datasets/vendors.csv', HEADER=TRUE);

CREATE OR REPLACE VIEW store_promotions AS
SELECT * FROM read_csv_auto('/content/datasets/store_promotions.csv', HEADER=TRUE);
""")

print("Vistas creadas correctamente")

Vistas creadas correctamente


## Pregunta 1 — Estacionalidad por formato

**Pregunta:**  
¿Cómo evoluciona el GMV semanal por formato de tienda? ¿Qué formato es más sensible a la estacionalidad? Identifica los 3 picos y las 3 caídas más significativas del período y propone una hipótesis para cada uno.

**Objetivo:**  
Construir una serie semanal de GMV por formato, medir su volatilidad relativa y detectar semanas con desviaciones relevantes para identificar patrones de estacionalidad y posibles explicaciones de negocio.

In [ ]:
query_1 = """

-- GMV semanal por formato
WITH ventas_semanales AS (
    SELECT
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)) AS week_start,
        s.format,
        SUM(t.total_amount) AS weekly_gmv
    FROM transactions t
    INNER JOIN stores s
        ON t.store_id = s.store_id
    GROUP BY
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)),
        s.format
)
SELECT
    week_start,
    format,
    ROUND(weekly_gmv, 2) AS weekly_gmv
FROM ventas_semanales
ORDER BY week_start, format;
"""

resultado_q1 = con.execute(query_1).df()
resultado_q1.head(20)

,week_start,format,weekly_gmv
0,2024-01-01,DESCUENTO,103479.67
1,2024-01-01,EXPRESS,23327.50
2,2024-01-01,HIPERMERCADO,212671.66
3,2024-01-01,SUPERMERCADO,208278.84
4,2024-01-08,DESCUENTO,110883.33
5,2024-01-08,EXPRESS,15342.10
6,2024-01-08,HIPERMERCADO,202598.31
7,2024-01-08,SUPERMERCADO,230940.74
8,2024-01-15,DESCUENTO,97260.05
9,2024-01-15,EXPRESS,23285.91


In [ ]:
query_1_1 = """

-- Sensibilidad a estacionalidad por formato usando coeficiente de variación
WITH ventas_semanales AS (
    SELECT
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)) AS week_start,
        s.format,
        SUM(t.total_amount) AS weekly_gmv
    FROM transactions t
    INNER JOIN stores s
        ON t.store_id = s.store_id
    GROUP BY
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)),
        s.format
),
resumen AS (
    SELECT
        format,
        AVG(weekly_gmv) AS avg_weekly_gmv,
        STDDEV_SAMP(weekly_gmv) AS std_weekly_gmv
    FROM ventas_semanales
    GROUP BY format
)
SELECT
    format,
    ROUND(avg_weekly_gmv, 2) AS avg_weekly_gmv,
    ROUND(std_weekly_gmv, 2) AS std_weekly_gmv,
    ROUND(std_weekly_gmv / NULLIF(avg_weekly_gmv, 0), 4) AS coef_variacion
FROM resumen
ORDER BY coef_variacion DESC;
"""

resultado_q1 = con.execute(query_1_1).df()
resultado_q1.head(20)

,format,avg_weekly_gmv,std_weekly_gmv,coef_variacion
0,EXPRESS,28252.97,6809.61,0.2410
1,DESCUENTO,116050.82,19024.14,0.1639
2,SUPERMERCADO,235560.84,36217.17,0.1537
3,HIPERMERCADO,236834.89,36133.01,0.1526


In [ ]:
query_1_2 = """

-- Top picos y caídas vs promedio móvil de 4 semanas previas
WITH ventas_semanales AS (
    SELECT
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)) AS week_start,
        s.format,
        SUM(t.total_amount) AS weekly_gmv
    FROM transactions t
    INNER JOIN stores s
        ON t.store_id = s.store_id
    GROUP BY
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)),
        s.format
),
serie AS (
    SELECT
        week_start,
        format,
        weekly_gmv,
        AVG(weekly_gmv) OVER (
            PARTITION BY format
            ORDER BY week_start
            ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING
        ) AS prev_4wk_avg
    FROM ventas_semanales
),
desvios AS (
    SELECT
        week_start,
        format,
        weekly_gmv,
        prev_4wk_avg,
        weekly_gmv - prev_4wk_avg AS abs_diff,
        ((weekly_gmv - prev_4wk_avg) * 100.0) / NULLIF(prev_4wk_avg, 0) AS pct_diff
    FROM serie
    WHERE prev_4wk_avg IS NOT NULL
)
SELECT
    week_start,
    format,
    ROUND(weekly_gmv, 2) AS weekly_gmv,
    ROUND(prev_4wk_avg, 2) AS prev_4wk_avg,
    ROUND(abs_diff, 2) AS abs_diff,
    ROUND(pct_diff, 2) AS pct_diff
FROM desvios
ORDER BY abs_diff DESC;
"""

resultado_q1 = con.execute(query_1_2).df()
resultado_q1.head(20)

,week_start,format,weekly_gmv,prev_4wk_avg,abs_diff,pct_diff
0,2024-12-02,HIPERMERCADO,335298.93,267024.54,68274.39,25.57
1,2024-12-09,SUPERMERCADO,333268.47,286233.93,47034.54,16.43
2,2024-12-16,HIPERMERCADO,341334.12,300394.78,40939.34,13.63
3,2024-11-18,SUPERMERCADO,294980.04,254436.61,40543.44,15.93
4,2024-07-01,SUPERMERCADO,260116.61,222367.49,37749.12,16.98
5,2025-03-17,SUPERMERCADO,266975.83,229655.63,37320.20,16.25
6,2024-07-22,SUPERMERCADO,279434.10,242181.86,37252.24,15.38
7,2024-03-04,HIPERMERCADO,233214.23,196732.40,36481.83,18.54
8,2024-09-30,HIPERMERCADO,258478.25,223053.08,35425.17,15.88
9,2025-05-05,SUPERMERCADO,268438.30,233152.99,35285.31,15.13



**Conclusión:**  
La estacionalidad no afecta por igual a todos los formatos. Este análisis permite identificar qué formatos requieren una planeación más fina de inventario, promociones y abastecimiento en semanas de alta y baja demanda.

## Pregunta 2 — Pareto de categorías por formato

**Pregunta:**  
¿Qué categorías concentran el 80% del GMV? ¿Las categorías líderes en HIPERMERCADO son las mismas que en DESCUENTO? ¿Qué dice esto sobre el perfil del comprador de cada formato?

**Objetivo:**  
Calcular la concentración de GMV por categoría dentro de cada formato, identificar el Pareto 80/20 y comparar si la composición comercial cambia entre formatos o si existe una mezcla similar de categorías líderes.

In [ ]:
query_2 = """

-- GMV por categoría y formato
WITH ventas_categoria AS (
    SELECT
        s.format,
        p.category,
        SUM(ti.unit_price * ti.quantity) AS gmv
    FROM transaction_items ti
    INNER JOIN transactions t
        ON ti.transaction_id = t.transaction_id
    INNER JOIN stores s
        ON t.store_id = s.store_id
    INNER JOIN products p
        ON ti.item_id = p.item_id
    GROUP BY
        s.format,
        p.category
)
SELECT
    format,
    category,
    ROUND(gmv, 2) AS gmv
FROM ventas_categoria
ORDER BY format, gmv DESC;
"""

resultado_q2 = con.execute(query_2).df()
resultado_q2.head(20)

,format,category,gmv
0,DESCUENTO,Electrónica,4808101.01
1,DESCUENTO,Hogar,2132633.80
2,DESCUENTO,Ropa,729587.68
3,DESCUENTO,Alimentos,511911.33
4,DESCUENTO,Juguetes,479405.64
5,DESCUENTO,Cuidado Personal,230341.26
6,DESCUENTO,Bebidas,157876.69
7,DESCUENTO,Limpieza,123724.05
8,EXPRESS,Electrónica,1164151.21
9,EXPRESS,Hogar,534369.90


In [ ]:
query_2_2 = """

-- Pareto acumulado por categoría dentro de cada formato
WITH ventas_categoria AS (
    SELECT
        s.format,
        p.category,
        SUM(ti.unit_price * ti.quantity) AS gmv
    FROM transaction_items ti
    INNER JOIN transactions t
        ON ti.transaction_id = t.transaction_id
    INNER JOIN stores s
        ON t.store_id = s.store_id
    INNER JOIN products p
        ON ti.item_id = p.item_id
    GROUP BY
        s.format,
        p.category
),
pareto AS (
    SELECT
        format,
        category,
        gmv,
        SUM(gmv) OVER (
            PARTITION BY format
            ORDER BY gmv DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_gmv,
        SUM(gmv) OVER (PARTITION BY format) AS total_gmv
    FROM ventas_categoria
)
SELECT
    format,
    category,
    ROUND(gmv, 2) AS gmv,
    ROUND(cumulative_gmv, 2) AS cumulative_gmv,
    ROUND((cumulative_gmv * 100.0) / NULLIF(total_gmv, 0), 2) AS cumulative_pct
FROM pareto
ORDER BY format, gmv DESC;
"""

resultado_q2 = con.execute(query_2_2).df()
resultado_q2.head(20)

,format,category,gmv,cumulative_gmv,cumulative_pct
0,DESCUENTO,Electrónica,4808101.01,4808101.01,52.41
1,DESCUENTO,Hogar,2132633.80,6940734.81,75.66
2,DESCUENTO,Ropa,729587.68,7670322.49,83.61
3,DESCUENTO,Alimentos,511911.33,8182233.82,89.19
4,DESCUENTO,Juguetes,479405.64,8661639.46,94.42
5,DESCUENTO,Cuidado Personal,230341.26,8891980.72,96.93
6,DESCUENTO,Bebidas,157876.69,9049857.41,98.65
7,DESCUENTO,Limpieza,123724.05,9173581.46,100.00
8,EXPRESS,Electrónica,1164151.21,1164151.21,52.12
9,EXPRESS,Hogar,534369.90,1698521.11,76.04


**Conclusión:**  
El mix comercial está concentrado en pocas categorías clave. Este resultado es útil para priorizar surtido, promociones y espacio en anaquel, y para evaluar si cada formato está realmente diferenciado desde la perspectiva de la demanda.

## Pregunta 3 — Cohortes de lealtad

**Pregunta:**  
Con base en la Query 3 del Bloque 1: ¿las cohortes más recientes retienen mejor o peor que las antiguas? ¿El ticket promedio de los clientes retenidos crece con el tiempo? ¿En qué mes se da la mayor caída y qué hipótesis tienes?

**Objetivo:**  
Analizar el comportamiento de retención de clientes loyalty por cohorte mensual, comparar cohortes antiguas vs recientes y evaluar cómo evoluciona el ticket promedio entre clientes retenidos a lo largo del tiempo.

In [ ]:
query_3 = """

-- Cohortes, retención y ticket promedio por mes desde la cohorte

WITH loyalty_transactions AS (
    SELECT
        customer_id,
        transaction_id,
        CAST(transaction_date AS DATE) AS transaction_date,
        DATE_TRUNC('month', CAST(transaction_date AS DATE)) AS transaction_month,
        total_amount
    FROM transactions
    WHERE loyalty_card = TRUE
      AND customer_id IS NOT NULL
),
first_purchase AS (
    SELECT
        customer_id,
        MIN(transaction_month) AS cohort_month
    FROM loyalty_transactions
    GROUP BY customer_id
),
cohort_base AS (
    SELECT
        lt.customer_id,
        fp.cohort_month,
        lt.transaction_month,
        lt.total_amount,
        (
            EXTRACT(YEAR FROM lt.transaction_month) * 12 + EXTRACT(MONTH FROM lt.transaction_month)
            - EXTRACT(YEAR FROM fp.cohort_month) * 12 - EXTRACT(MONTH FROM fp.cohort_month)
        ) AS month_number
    FROM loyalty_transactions lt
    INNER JOIN first_purchase fp
        ON lt.customer_id = fp.customer_id
),
cohort_size AS (
    SELECT
        cohort_month,
        COUNT(DISTINCT customer_id) AS cohort_size
    FROM first_purchase
    GROUP BY cohort_month
),
activity AS (
    SELECT
        cohort_month,
        month_number,
        COUNT(DISTINCT customer_id) AS active_customers,
        AVG(total_amount) AS avg_ticket
    FROM cohort_base
    GROUP BY cohort_month, month_number
)
SELECT
    a.cohort_month,
    c.cohort_size,
    a.month_number,
    a.active_customers,
    ROUND((a.active_customers * 100.0) / NULLIF(c.cohort_size, 0), 2) AS retention_pct,
    ROUND(a.avg_ticket, 2) AS avg_ticket
FROM activity a
INNER JOIN cohort_size c
    ON a.cohort_month = c.cohort_month
ORDER BY a.cohort_month, a.month_number;
"""

resultado_q3 = con.execute(query_3).df()
resultado_q3.head(20)

,cohort_month,cohort_size,month_number,active_customers,retention_pct,avg_ticket
0,2024-01-01,2076,0,2076,100.00,287.61
1,2024-01-01,2076,1,1334,64.26,285.90
2,2024-01-01,2076,2,1414,68.11,291.59
3,2024-01-01,2076,3,1377,66.33,281.80
4,2024-01-01,2076,4,1508,72.64,286.96
5,2024-01-01,2076,5,1500,72.25,280.04
6,2024-01-01,2076,6,1572,75.72,276.21
7,2024-01-01,2076,7,1515,72.98,276.24
8,2024-01-01,2076,8,1461,70.38,274.77
9,2024-01-01,2076,9,1548,74.57,284.30


In [ ]:
query_3_3 = """

-- Cohortes, retención y ticket promedio por mes desde la cohorte

-- Vista pivoteada para comparación de cohortes
WITH cohort_metrics AS (
    WITH loyalty_transactions AS (
        SELECT
            customer_id,
            transaction_id,
            CAST(transaction_date AS DATE) AS transaction_date,
            DATE_TRUNC('month', CAST(transaction_date AS DATE)) AS transaction_month,
            total_amount
        FROM transactions
        WHERE loyalty_card = TRUE
          AND customer_id IS NOT NULL
    ),
    first_purchase AS (
        SELECT
            customer_id,
            MIN(transaction_month) AS cohort_month
        FROM loyalty_transactions
        GROUP BY customer_id
    ),
    cohort_base AS (
        SELECT
            lt.customer_id,
            fp.cohort_month,
            lt.transaction_month,
            lt.total_amount,
            (
                EXTRACT(YEAR FROM lt.transaction_month) * 12 + EXTRACT(MONTH FROM lt.transaction_month)
                - EXTRACT(YEAR FROM fp.cohort_month) * 12 - EXTRACT(MONTH FROM fp.cohort_month)
            ) AS month_number
        FROM loyalty_transactions lt
        INNER JOIN first_purchase fp
            ON lt.customer_id = fp.customer_id
    ),
    cohort_size AS (
        SELECT
            cohort_month,
            COUNT(DISTINCT customer_id) AS cohort_size
        FROM first_purchase
        GROUP BY cohort_month
    ),
    activity AS (
        SELECT
            cohort_month,
            month_number,
            COUNT(DISTINCT customer_id) AS active_customers,
            AVG(total_amount) AS avg_ticket
        FROM cohort_base
        GROUP BY cohort_month, month_number
    )
    SELECT
        a.cohort_month,
        c.cohort_size,
        a.month_number,
        ROUND((a.active_customers * 100.0) / NULLIF(c.cohort_size, 0), 2) AS retention_pct,
        ROUND(a.avg_ticket, 2) AS avg_ticket
    FROM activity a
    INNER JOIN cohort_size c
        ON a.cohort_month = c.cohort_month
)
SELECT
    cohort_month,
    MAX(cohort_size) AS cohort_size,
    MAX(CASE WHEN month_number = 1 THEN retention_pct END) AS retention_m1,
    MAX(CASE WHEN month_number = 2 THEN retention_pct END) AS retention_m2,
    MAX(CASE WHEN month_number = 3 THEN retention_pct END) AS retention_m3,
    MAX(CASE WHEN month_number = 6 THEN retention_pct END) AS retention_m6,
    MAX(CASE WHEN month_number = 0 THEN avg_ticket END) AS avg_ticket_m0,
    MAX(CASE WHEN month_number = 1 THEN avg_ticket END) AS avg_ticket_m1,
    MAX(CASE WHEN month_number = 2 THEN avg_ticket END) AS avg_ticket_m2,
    MAX(CASE WHEN month_number = 3 THEN avg_ticket END) AS avg_ticket_m3,
    MAX(CASE WHEN month_number = 6 THEN avg_ticket END) AS avg_ticket_m6
FROM cohort_metrics
GROUP BY cohort_month
ORDER BY cohort_month;
"""

resultado_q3 = con.execute(query_3_3).df()
resultado_q3.head(20)

,cohort_month,cohort_size,retention_m1,retention_m2,retention_m3,retention_m6,avg_ticket_m0,avg_ticket_m1,avg_ticket_m2,avg_ticket_m3,avg_ticket_m6
0,2024-01-01,2076,64.26,68.11,66.33,75.72,287.61,285.90,291.59,281.80,276.21
1,2024-02-01,598,69.23,68.56,71.91,71.74,274.49,299.78,284.37,284.09,259.78
2,2024-03-01,213,68.54,69.95,78.40,75.12,288.24,287.86,253.71,287.43,265.87
3,2024-04-01,66,77.27,66.67,77.27,83.33,303.07,293.03,265.65,260.40,299.89
4,2024-05-01,30,70.00,76.67,73.33,70.00,312.01,350.06,252.98,261.52,239.38
5,2024-06-01,14,78.57,57.14,71.43,85.71,145.73,113.02,386.79,270.71,291.80
6,2024-07-01,1,100.00,100.00,100.00,100.00,115.30,83.85,122.33,77.96,102.44
7,2024-08-01,2,100.00,50.00,50.00,NaN,214.78,279.45,923.99,803.36,NaN


**Conclusión:**  
El principal valor de este análisis es identificar en qué momento del ciclo del cliente se pierde más retención y si el valor económico de los clientes retenidos mejora o se deteriora con el tiempo.

## Pregunta 4 — Quiebres de stock y su impacto

**Pregunta:**  
Con los gaps detectados en la Query 5: ¿hay categorías o proveedores donde los quiebres son sistemáticos? ¿Cuánto GMV total estimado se perdió? ¿Es un problema de demanda o de abastecimiento?

**Objetivo:**  
Tomar la base de posibles gaps de venta y analizar su concentración por categoría y proveedor para estimar el impacto económico y formular una hipótesis sobre si el fenómeno responde a desabastecimiento o a demanda intermitente.

In [ ]:
query_4 = """

-- Base de posibles quiebres de stock

WITH ventas_diarias AS (
    SELECT
        t.store_id,
        s.store_name,
        ti.item_id,
        p.item_name,
        p.category,
        CAST(t.transaction_date AS DATE) AS sale_date,
        SUM(ti.quantity) AS qty_sold,
        SUM(ti.unit_price * ti.quantity) AS gmv_sold
    FROM transaction_items ti
    INNER JOIN transactions t
        ON ti.transaction_id = t.transaction_id
    INNER JOIN stores s
        ON t.store_id = s.store_id
    INNER JOIN products p
        ON ti.item_id = p.item_id
    GROUP BY
        t.store_id,
        s.store_name,
        ti.item_id,
        p.item_name,
        p.category,
        CAST(t.transaction_date AS DATE)
),
rangos_tienda_item AS (
    SELECT
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        MIN(sale_date) AS min_sale_date,
        MAX(sale_date) AS max_sale_date
    FROM ventas_diarias
    GROUP BY
        store_id,
        store_name,
        item_id,
        item_name,
        category
),
calendario_tienda_item AS (
    SELECT
        r.store_id,
        r.store_name,
        r.item_id,
        r.item_name,
        r.category,
        gs.generate_series AS calendar_date
    FROM rangos_tienda_item r,
    generate_series(r.min_sale_date, r.max_sale_date, INTERVAL '1 day') AS gs
),
base_completa AS (
    SELECT
        c.store_id,
        c.store_name,
        c.item_id,
        c.item_name,
        c.category,
        c.calendar_date,
        COALESCE(v.qty_sold, 0) AS qty_sold,
        COALESCE(v.gmv_sold, 0) AS gmv_sold
    FROM calendario_tienda_item c
    LEFT JOIN ventas_diarias v
        ON c.store_id = v.store_id
       AND c.item_id = v.item_id
       AND c.calendar_date = v.sale_date
),
dias_sin_venta AS (
    SELECT
        *,
        calendar_date
          - INTERVAL '1 day' * ROW_NUMBER() OVER (
                PARTITION BY store_id, item_id
                ORDER BY calendar_date
            ) AS grp_gap
    FROM base_completa
    WHERE qty_sold = 0
),
gaps_consecutivos AS (
    SELECT
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        MIN(calendar_date) AS gap_start_date,
        MAX(calendar_date) AS gap_end_date,
        COUNT(*) AS gap_days
    FROM dias_sin_venta
    GROUP BY
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        grp_gap
),
gaps_filtrados AS (
    SELECT *
    FROM gaps_consecutivos
    WHERE gap_days >= 3
),
ventas_previas AS (
    SELECT
        g.store_id,
        g.item_id,
        g.gap_start_date,
        AVG(b.gmv_sold) AS avg_daily_gmv_before_gap
    FROM gaps_filtrados g
    INNER JOIN base_completa b
        ON g.store_id = b.store_id
       AND g.item_id = b.item_id
       AND b.calendar_date BETWEEN g.gap_start_date - INTERVAL '30 day' AND g.gap_start_date - INTERVAL '1 day'
       AND b.gmv_sold > 0
    GROUP BY
        g.store_id,
        g.item_id,
        g.gap_start_date
),
resultado_final_q5 AS (
    SELECT
        g.store_id,
        g.store_name,
        g.item_id,
        g.item_name,
        g.category,
        g.gap_start_date,
        g.gap_end_date,
        g.gap_days,
        ROUND(vp.avg_daily_gmv_before_gap, 2) AS avg_daily_gmv_before_gap,
        ROUND(vp.avg_daily_gmv_before_gap * g.gap_days, 2) AS estimated_gmv_lost
    FROM gaps_filtrados g
    LEFT JOIN ventas_previas vp
        ON g.store_id = vp.store_id
       AND g.item_id = vp.item_id
       AND g.gap_start_date = vp.gap_start_date
)
SELECT *
FROM resultado_final_q5
ORDER BY estimated_gmv_lost DESC NULLS LAST;
"""

resultado_q4 = con.execute(query_4).df()
resultado_q4.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,store_id,store_name,item_id,item_name,category,gap_start_date,gap_end_date,gap_days,avg_daily_gmv_before_gap,estimated_gmv_lost
0,TIENDA_039,Tienda San Salvador 39,ITEM_069,Electrónica Producto 69,Electrónica,2024-07-04,2024-09-09,68,1784.29,121331.38
1,TIENDA_029,Tienda San Salvador 29,ITEM_091,Electrónica Producto 91,Electrónica,2024-12-26,2025-03-10,75,1549.81,116235.56
2,TIENDA_036,Tienda Heredia 36,ITEM_007,Electrónica Producto 7,Electrónica,2025-01-01,2025-04-05,95,1105.56,105028.20
3,TIENDA_024,Tienda Santa Ana 24,ITEM_123,Electrónica Producto 123,Electrónica,2024-10-06,2024-12-18,74,1357.20,100432.80
4,TIENDA_035,Tienda Masaya 35,ITEM_130,Electrónica Producto 130,Electrónica,2024-03-21,2024-05-28,69,1214.22,83781.52
5,TIENDA_030,Tienda Masaya 30,ITEM_123,Electrónica Producto 123,Electrónica,2024-03-15,2024-05-09,56,1489.53,83413.68
6,TIENDA_029,Tienda San Salvador 29,ITEM_123,Electrónica Producto 123,Electrónica,2024-01-08,2024-03-02,55,1455.54,80054.70
7,TIENDA_035,Tienda Masaya 35,ITEM_091,Electrónica Producto 91,Electrónica,2025-01-07,2025-03-24,77,989.35,76179.95
8,TIENDA_040,Tienda Managua 40,ITEM_007,Electrónica Producto 7,Electrónica,2025-02-22,2025-05-11,79,954.33,75391.81
9,TIENDA_036,Tienda Heredia 36,ITEM_096,Electrónica Producto 96,Electrónica,2024-08-13,2024-11-27,107,700.89,74995.66


In [ ]:
query_q5_base = """
WITH ventas_diarias AS (
    SELECT
        t.store_id,
        s.store_name,
        ti.item_id,
        p.item_name,
        p.category,
        CAST(t.transaction_date AS DATE) AS sale_date,
        SUM(ti.quantity) AS qty_sold,
        SUM(ti.unit_price * ti.quantity) AS gmv_sold
    FROM transaction_items ti
    INNER JOIN transactions t
        ON ti.transaction_id = t.transaction_id
    INNER JOIN stores s
        ON t.store_id = s.store_id
    INNER JOIN products p
        ON ti.item_id = p.item_id
    GROUP BY
        t.store_id,
        s.store_name,
        ti.item_id,
        p.item_name,
        p.category,
        CAST(t.transaction_date AS DATE)
),
rangos_tienda_item AS (
    SELECT
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        MIN(sale_date) AS min_sale_date,
        MAX(sale_date) AS max_sale_date
    FROM ventas_diarias
    GROUP BY
        store_id,
        store_name,
        item_id,
        item_name,
        category
),
calendario_tienda_item AS (
    SELECT
        r.store_id,
        r.store_name,
        r.item_id,
        r.item_name,
        r.category,
        gs.generate_series AS calendar_date
    FROM rangos_tienda_item r,
    generate_series(r.min_sale_date, r.max_sale_date, INTERVAL '1 day') AS gs
),
base_completa AS (
    SELECT
        c.store_id,
        c.store_name,
        c.item_id,
        c.item_name,
        c.category,
        c.calendar_date,
        COALESCE(v.qty_sold, 0) AS qty_sold,
        COALESCE(v.gmv_sold, 0) AS gmv_sold
    FROM calendario_tienda_item c
    LEFT JOIN ventas_diarias v
        ON c.store_id = v.store_id
       AND c.item_id = v.item_id
       AND c.calendar_date = v.sale_date
),
dias_sin_venta AS (
    SELECT
        *,
        calendar_date
          - INTERVAL '1 day' * ROW_NUMBER() OVER (
                PARTITION BY store_id, item_id
                ORDER BY calendar_date
            ) AS grp_gap
    FROM base_completa
    WHERE qty_sold = 0
),
gaps_consecutivos AS (
    SELECT
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        MIN(calendar_date) AS gap_start_date,
        MAX(calendar_date) AS gap_end_date,
        COUNT(*) AS gap_days
    FROM dias_sin_venta
    GROUP BY
        store_id,
        store_name,
        item_id,
        item_name,
        category,
        grp_gap
),
gaps_filtrados AS (
    SELECT *
    FROM gaps_consecutivos
    WHERE gap_days >= 3
),
ventas_previas AS (
    SELECT
        g.store_id,
        g.item_id,
        g.gap_start_date,
        AVG(b.gmv_sold) AS avg_daily_gmv_before_gap
    FROM gaps_filtrados g
    INNER JOIN base_completa b
        ON g.store_id = b.store_id
       AND g.item_id = b.item_id
       AND b.calendar_date BETWEEN g.gap_start_date - INTERVAL '30 day' AND g.gap_start_date - INTERVAL '1 day'
       AND b.gmv_sold > 0
    GROUP BY
        g.store_id,
        g.item_id,
        g.gap_start_date
)
SELECT
    g.store_id,
    g.store_name,
    g.item_id,
    g.item_name,
    g.category,
    g.gap_start_date,
    g.gap_end_date,
    g.gap_days,
    ROUND(vp.avg_daily_gmv_before_gap, 2) AS avg_daily_gmv_before_gap,
    ROUND(vp.avg_daily_gmv_before_gap * g.gap_days, 2) AS estimated_gmv_lost
FROM gaps_filtrados g
LEFT JOIN ventas_previas vp
    ON g.store_id = vp.store_id
   AND g.item_id = vp.item_id
   AND g.gap_start_date = vp.gap_start_date
"""

con.execute(f"CREATE OR REPLACE TEMP VIEW resultado_final_q5 AS {query_q5_base}")
print("Vista temporal resultado_final_q5 creada correctamente")

Vista temporal resultado_final_q5 creada correctamente


In [ ]:
query_4_4 = """

-- Impacto de quiebres por categoría

SELECT
    category,
    COUNT(*) AS total_gaps,
    ROUND(SUM(estimated_gmv_lost), 2) AS total_estimated_gmv_lost,
    ROUND(AVG(gap_days), 2) AS avg_gap_days
FROM resultado_final_q5
GROUP BY category
ORDER BY total_estimated_gmv_lost DESC;
"""

resultado_q4 = con.execute(query_4_4).df()
resultado_q4.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category,total_gaps,total_estimated_gmv_lost,avg_gap_days
0,Electrónica,31417,1.779814e+08,11.23
1,Hogar,62516,7.907496e+07,11.31
2,Ropa,33137,2.666157e+07,11.18
3,Alimentos,67261,1.886009e+07,11.27
4,Juguetes,28326,1.734888e+07,11.23
5,Cuidado Personal,32505,8.353526e+06,11.39
6,Bebidas,35963,5.758288e+06,11.27
7,Limpieza,21910,4.505177e+06,11.23


In [ ]:
query_4_5 = """

-- Impacto de quiebres por proveedor

WITH gaps_enriquecidos AS (
    SELECT
        g.*,
        p.vendor_id,
        v.vendor_name
    FROM resultado_final_q5 g
    INNER JOIN products p
        ON g.item_id = p.item_id
    INNER JOIN vendors v
        ON p.vendor_id = v.vendor_id
)
SELECT
    vendor_id,
    vendor_name,
    COUNT(*) AS total_gaps,
    ROUND(SUM(estimated_gmv_lost), 2) AS total_estimated_gmv_lost,
    ROUND(AVG(gap_days), 2) AS avg_gap_days
FROM gaps_enriquecidos
GROUP BY
    vendor_id,
    vendor_name
ORDER BY total_estimated_gmv_lost DESC;
"""

resultado_q4_proveedor = con.execute(query_4_5).df()
resultado_q4_proveedor.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,vendor_id,vendor_name,total_gaps,total_estimated_gmv_lost,avg_gap_days
0,VND_020,Proveedor T,15574,27695417.05,11.32
1,VND_026,Proveedor Z,9420,25255646.74,11.22
2,VND_010,Proveedor J,14150,24477053.01,11.24
3,VND_003,Proveedor C,10974,23962077.73,11.31
4,VND_027,Proveedor Z1,14029,23835185.39,11.32
5,VND_004,Proveedor D,12520,21476125.94,11.26
6,VND_011,Proveedor K,9457,20076715.53,11.17
7,VND_016,Proveedor P,17281,19894665.78,11.19
8,VND_021,Proveedor U,7745,18748294.15,11.37
9,VND_019,Proveedor S,10951,16073075.91,11.28


**Conclusión:**  
Esta query es útil como sistema de priorización operativa para identificar dónde revisar surtido, abastecimiento y reposición antes de asumir que toda ausencia de venta es un quiebre real.

## Pregunta 5 — Hallazgo libre

**Pregunta:**  
Identifica por tu cuenta un hallazgo relevante que no estaba en las preguntas anteriores. Muéstralo con evidencia y explica el impacto para el negocio.

**Objetivo:**  
Explorar una hipótesis adicional con impacto potencial para negocio. En este caso, se evalúa si una promoción tipo experimento (`CONTROL` vs `TREATMENT`) parece generar uplift en GMV, transacciones o ticket promedio.

In [ ]:
query_5 = """

-- Comparación de una promoción entre CONTROL vs TREATMENT
WITH promo_period AS (
    SELECT
        promo_name,
        variant,
        CAST(start_date AS DATE) AS start_date,
        CAST(end_date AS DATE) AS end_date,
        store_id
    FROM store_promotions
    WHERE promo_name = 'Exhibicion_Q3_2024'
),
ventas_promo AS (
    SELECT
        sp.promo_name,
        sp.variant,
        'PROMO_PERIOD' AS period_type,
        SUM(t.total_amount) AS gmv,
        COUNT(DISTINCT t.transaction_id) AS transactions,
        AVG(t.total_amount) AS avg_ticket
    FROM promo_period sp
    INNER JOIN transactions t
        ON sp.store_id = t.store_id
       AND CAST(t.transaction_date AS DATE) BETWEEN sp.start_date AND sp.end_date
    GROUP BY sp.promo_name, sp.variant
),
ventas_pre AS (
    SELECT
        sp.promo_name,
        sp.variant,
        'PRE_PERIOD' AS period_type,
        SUM(t.total_amount) AS gmv,
        COUNT(DISTINCT t.transaction_id) AS transactions,
        AVG(t.total_amount) AS avg_ticket
    FROM promo_period sp
    INNER JOIN transactions t
        ON sp.store_id = t.store_id
       AND CAST(t.transaction_date AS DATE)
           BETWEEN sp.start_date - INTERVAL '42 day' AND sp.start_date - INTERVAL '1 day'
    GROUP BY sp.promo_name, sp.variant
)
SELECT * FROM ventas_pre
UNION ALL
SELECT * FROM ventas_promo
ORDER BY promo_name, variant, period_type;
"""

resultado_q5 = con.execute(query_5).df()
resultado_q5.head(20)

,promo_name,variant,period_type,gmv,transactions,avg_ticket
0,Exhibicion_Q3_2024,CONTROL,PRE_PERIOD,2156853.59,7845,274.933536
1,Exhibicion_Q3_2024,CONTROL,PROMO_PERIOD,2031997.58,7262,279.812391
2,Exhibicion_Q3_2024,TREATMENT,PRE_PERIOD,1816044.15,6561,276.793804
3,Exhibicion_Q3_2024,TREATMENT,PROMO_PERIOD,1899112.65,6882,275.953596


**Conclusión:**  
Este hallazgo libre aporta valor porque conecta el análisis exploratorio con una posible decisión comercial concreta.

## Parte B — Interpretación de A/B Test

### Pregunta 1 — Validación del experimento

**Pregunta:**  
¿Los grupos son comparables en GMV base, formato y tamaño antes del test? ¿Hay alguna tienda asignada a ambos grupos?

**Objetivo:**  
Validar si la asignación experimental entre `CONTROL` y `TREATMENT` es razonablemente balanceada antes del inicio del experimento.  
Esto es importante porque, si los grupos no son comparables en su nivel base, cualquier diferencia posterior podría deberse al punto de partida y no al efecto real de la nueva exhibición.

In [ ]:
query = """

-- Base del experimento: tiendas participantes en la promo de exhibición Q3 2024

WITH experimento AS (
    SELECT
        store_id,
        promo_name,
        variant,
        CAST(start_date AS DATE) AS start_date,
        CAST(end_date AS DATE) AS end_date
    FROM store_promotions
    WHERE promo_name = 'Exhibicion_Q3_2024'
),

asignacion_tiendas AS (
    SELECT DISTINCT
        e.store_id,
        e.variant,
        e.start_date,
        e.end_date,
        s.store_name,
        s.format,
        s.size_sqm,
        s.country,
        s.region
    FROM experimento e
    INNER JOIN stores s
        ON e.store_id = s.store_id
)

SELECT
    *
FROM asignacion_tiendas
ORDER BY variant, store_id;
"""
df = con.execute(query).df()
df.head(20)

,store_id,variant,start_date,end_date,store_name,format,size_sqm,country,region
0,TIENDA_001,CONTROL,2024-09-01,2024-10-12,Tienda San José 1,HIPERMERCADO,7037,CR,Sur
1,TIENDA_002,CONTROL,2024-09-01,2024-10-12,Tienda Guatemala City 2,HIPERMERCADO,7016,GT,Capital
2,TIENDA_003,CONTROL,2024-09-01,2024-10-12,Tienda La Ceiba 3,HIPERMERCADO,4356,HN,Occidente
3,TIENDA_006,CONTROL,2024-09-01,2024-10-12,Tienda Cartago 6,HIPERMERCADO,5839,CR,Occidente
4,TIENDA_007,CONTROL,2024-09-01,2024-10-12,Tienda Guatemala City 7,HIPERMERCADO,6859,GT,Oriente
5,TIENDA_008,CONTROL,2024-09-01,2024-10-12,Tienda San Pedro Sula 8,HIPERMERCADO,4881,HN,Sur
6,TIENDA_009,CONTROL,2024-09-01,2024-10-12,Tienda San Salvador 9,SUPERMERCADO,1698,SV,Sur
7,TIENDA_010,CONTROL,2024-09-01,2024-10-12,Tienda Masaya 10,SUPERMERCADO,3152,NI,Capital
8,TIENDA_011,CONTROL,2024-09-01,2024-10-12,Tienda San José 11,SUPERMERCADO,1661,CR,Occidente
9,TIENDA_012,CONTROL,2024-09-01,2024-10-12,Tienda Escuintla 12,SUPERMERCADO,3313,GT,Sur


### Conclusión

Esta primera query construye la base del experimento:

- identifica las tiendas incluidas en la promoción / test
- recupera su asignación a `CONTROL` o `TREATMENT`
- agrega atributos descriptivos de tienda:
  - `format`
  - `size_sqm`
  - `country`
  - `region`

Con esto podemos revisar si ambos grupos tienen una composición parecida antes del test.

In [ ]:
query = """

-- Validación 1: ¿Hay tiendas asignadas a ambos grupos?

WITH experimento AS (
    SELECT
        store_id,
        variant
    FROM store_promotions
    WHERE promo_name = 'Exhibicion_Q3_2024'
),

conflictos AS (
    SELECT
        store_id,
        COUNT(DISTINCT variant) AS total_variants
    FROM experimento
    GROUP BY store_id
    HAVING COUNT(DISTINCT variant) > 1
)

SELECT
    *
FROM conflictos
ORDER BY store_id;
"""
df = con.execute(query).df()
df.head(20)

,store_id,total_variants
0,TIENDA_008,2
1,TIENDA_037,2


### Conclusión
Esta query revisa si alguna tienda aparece simultáneamente en más de una variante.

**Interpretación esperada:**
- si devuelve **0 filas**, no hay conflicto de asignación
- si devuelve filas, entonces hay tiendas asignadas tanto a `CONTROL` como a `TREATMENT`, lo cual compromete la validez del experimento

In [ ]:
query = """

-- Validación 2: comparabilidad en formato, tamaño y cantidad de tiendas por grupo

WITH experimento AS (
    SELECT DISTINCT
        sp.store_id,
        sp.variant
    FROM store_promotions sp
    WHERE sp.promo_name = 'Exhibicion_Q3_2024'
),

base_tiendas AS (
    SELECT
        e.variant,
        s.store_id,
        s.store_name,
        s.format,
        s.size_sqm
    FROM experimento e
    INNER JOIN stores s
        ON e.store_id = s.store_id
)

SELECT
    variant,
    format,
    COUNT(*) AS total_stores,
    ROUND(AVG(size_sqm), 2) AS avg_size_sqm,
    ROUND(MIN(size_sqm), 2) AS min_size_sqm,
    ROUND(MAX(size_sqm), 2) AS max_size_sqm
FROM base_tiendas
GROUP BY
    variant,
    format
ORDER BY
    variant,
    format;
"""
df = con.execute(query).df()
df.head(20)

,variant,format,total_stores,avg_size_sqm,min_size_sqm,max_size_sqm
0,CONTROL,DESCUENTO,4,1181.50,963,1691
1,CONTROL,EXPRESS,2,290.00,257,323
2,CONTROL,HIPERMERCADO,6,5998.00,4356,7037
3,CONTROL,SUPERMERCADO,8,2454.38,1661,3313
4,TREATMENT,DESCUENTO,8,1322.38,961,1794
5,TREATMENT,EXPRESS,4,437.00,257,537
6,TREATMENT,HIPERMERCADO,3,4457.33,4108,4881
7,TREATMENT,SUPERMERCADO,7,2241.86,1642,3294


### Conclusión

Esta query compara los grupos en variables estructurales:

- número de tiendas por grupo
- distribución por formato
- tamaño promedio de tienda (`size_sqm`)

La idea es verificar si `CONTROL` y `TREATMENT` parten de una composición parecida o si un grupo quedó sesgado hacia tiendas más grandes, más pequeñas o hacia un formato específico.

In [ ]:
query = """

-- Validación 3: comparabilidad en GMV base antes del test
-- Se usa una ventana de 6 semanas previas al inicio del experimento.

WITH experimento AS (
    SELECT DISTINCT
        sp.store_id,
        sp.variant,
        CAST(sp.start_date AS DATE) AS start_date
    FROM store_promotions sp
    WHERE sp.promo_name = 'Exhibicion_Q3_2024'
),

gmv_base AS (
    SELECT
        e.store_id,
        e.variant,
        SUM(t.total_amount) AS gmv_base_6w,
        COUNT(DISTINCT t.transaction_id) AS tx_base_6w,
        AVG(t.total_amount) AS avg_ticket_base_6w
    FROM experimento e
    INNER JOIN transactions t
        ON e.store_id = t.store_id
       AND CAST(t.transaction_date AS DATE)
           BETWEEN e.start_date - INTERVAL '42 day' AND e.start_date - INTERVAL '1 day'
    GROUP BY
        e.store_id,
        e.variant
)

SELECT
    variant,
    COUNT(*) AS total_stores,
    ROUND(AVG(gmv_base_6w), 2) AS avg_gmv_base_6w,
    ROUND(STDDEV_SAMP(gmv_base_6w), 2) AS std_gmv_base_6w,
    ROUND(AVG(tx_base_6w), 2) AS avg_tx_base_6w,
    ROUND(AVG(avg_ticket_base_6w), 2) AS avg_ticket_base_6w
FROM gmv_base
GROUP BY variant
ORDER BY variant;
"""
df = con.execute(query).df()
df.head(20)

,variant,total_stores,avg_gmv_base_6w,std_gmv_base_6w,avg_tx_base_6w,avg_ticket_base_6w
0,CONTROL,20,107842.68,52245.89,392.25,273.70
1,TREATMENT,22,82547.46,46538.37,298.23,273.96


### Conclusión

Esta query compara el desempeño base antes del test usando una ventana de **6 semanas previas**.

Mide por grupo:

- GMV base promedio por tienda
- desviación estándar del GMV base
- transacciones promedio
- ticket promedio base

Esto sirve para evaluar si el punto de partida de `CONTROL` y `TREATMENT` era razonablemente similar antes de aplicar la nueva exhibición.

### Cierre / conclusión

La validación del experimento mostró que:

- las tiendas asignadas a `CONTROL` y `TREATMENT` son **[comparables / no totalmente comparables]** en términos de formato y tamaño
- el GMV base previo al experimento fue **[similar / diferente]** entre grupos
- la revisión de asignación cruzada mostró **[0 / N]** tiendas en ambos grupos

**Conclusión:**  
El experimento parte de una base **[razonablemente balanceada / con señales de desbalance]**.  
Por lo tanto, el análisis posterior del efecto en GMV tendrá una interpretación **[más confiable / debe leerse con cautela]**.

**Comentario de negocio:**  
Si los grupos son comparables y no hay contaminación entre variantes, entonces las diferencias observadas durante el test pueden atribuirse con más confianza a la nueva estrategia de exhibición.

## Parte B — Interpretación de A/B Test

### Pregunta 2 — Resultado en GMV

**Pregunta:**  
¿El GMV promedio semanal por tienda en `TREATMENT` es significativamente mayor? Usa un t-test. Reporta: diferencia absoluta, lift relativo, p-value e intervalo de confianza al 95%.

**Objetivo:**  
Construir una base semanal por tienda durante el período del experimento y comparar estadísticamente el desempeño entre `CONTROL` y `TREATMENT`.  
La meta es determinar si la nueva exhibición tuvo un efecto positivo en GMV y si ese efecto es estadísticamente significativo.

In [ ]:
query_b2_gmv = """
WITH experimento AS (
    SELECT DISTINCT
        sp.store_id,
        sp.variant,
        CAST(sp.start_date AS DATE) AS start_date,
        CAST(sp.end_date AS DATE) AS end_date
    FROM store_promotions sp
    WHERE sp.promo_name = 'Exhibicion_Q3_2024'
),
ventas_semanales AS (
    SELECT
        e.store_id,
        e.variant,
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)) AS week_start,
        SUM(t.total_amount) AS weekly_gmv
    FROM experimento e
    INNER JOIN transactions t
        ON e.store_id = t.store_id
       AND CAST(t.transaction_date AS DATE) BETWEEN e.start_date AND e.end_date
    GROUP BY
        e.store_id,
        e.variant,
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE))
)
SELECT
    store_id,
    variant,
    week_start,
    weekly_gmv
FROM ventas_semanales
ORDER BY variant, store_id, week_start;
"""

df_gmv = con.execute(query_b2_gmv).df()
df_gmv.head()

,store_id,variant,week_start,weekly_gmv
0,TIENDA_001,CONTROL,2024-08-26,3819.98
1,TIENDA_001,CONTROL,2024-09-02,20770.31
2,TIENDA_001,CONTROL,2024-09-09,26038.98
3,TIENDA_001,CONTROL,2024-09-16,23099.66
4,TIENDA_001,CONTROL,2024-09-23,32731.74


Esta query arma la base mínima necesaria para el análisis estadístico:

- una fila por **tienda-semana**
- la variante asignada (`CONTROL` o `TREATMENT`)
- el `weekly_gmv` observado durante el test

Con esta estructura se puede comparar el promedio semanal por tienda entre ambos grupos.

### Paso estadístico

A partir de la base semanal por tienda, se compara el `weekly_gmv` de `TREATMENT` vs `CONTROL` con un **t-test de dos muestras independientes**.

Se reportarán:
- media de cada grupo
- diferencia absoluta
- lift relativo
- p-value
- intervalo de confianza al 95%

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Separar grupos
control = df_gmv[df_gmv["variant"] == "CONTROL"]["weekly_gmv"].dropna()
treatment = df_gmv[df_gmv["variant"] == "TREATMENT"]["weekly_gmv"].dropna()

# Estadísticos base
mean_control = control.mean()
mean_treatment = treatment.mean()

diff_abs = mean_treatment - mean_control
lift_rel = diff_abs / mean_control if mean_control != 0 else np.nan

# Welch t-test (más robusto si las varianzas no son iguales)
t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)

# Intervalo de confianza 95% para diferencia de medias (TREATMENT - CONTROL)
n_t = len(treatment)
n_c = len(control)

var_t = treatment.var(ddof=1)
var_c = control.var(ddof=1)

se_diff = np.sqrt((var_t / n_t) + (var_c / n_c))

# grados de libertad de Welch
df_welch = ((var_t / n_t + var_c / n_c) ** 2) / (
    ((var_t / n_t) ** 2) / (n_t - 1) + ((var_c / n_c) ** 2) / (n_c - 1)
)

t_crit = stats.t.ppf(0.975, df_welch)

ci_low = diff_abs - t_crit * se_diff
ci_high = diff_abs + t_crit * se_diff

resultados_gmv = pd.DataFrame({
    "metric": [
        "n_control",
        "n_treatment",
        "mean_control",
        "mean_treatment",
        "diff_abs",
        "lift_rel_pct",
        "t_stat",
        "p_value",
        "ci95_low",
        "ci95_high"
    ],
    "value": [
        n_c,
        n_t,
        mean_control,
        mean_treatment,
        diff_abs,
        lift_rel * 100,
        t_stat,
        p_value,
        ci_low,
        ci_high
    ]
})

resultados_gmv

,metric,value
0,n_control,140.000000
1,n_treatment,154.000000
2,mean_control,14514.268429
3,mean_treatment,12331.900325
4,diff_abs,-2182.368104
5,lift_rel_pct,-15.036019
6,t_stat,-2.089165
7,p_value,0.037570
8,ci95_low,-4238.406902
9,ci95_high,-126.329306


In [ ]:
# Resumen amigable para leer rápido
print("Observaciones CONTROL:", n_c)
print("Observaciones TREATMENT:", n_t)
print("GMV promedio semanal CONTROL:", round(mean_control, 2))
print("GMV promedio semanal TREATMENT:", round(mean_treatment, 2))
print("Diferencia absoluta:", round(diff_abs, 2))
print("Lift relativo:", round(lift_rel * 100, 2), "%")
print("p-value:", round(p_value, 6))
print("IC 95% diferencia: [", round(ci_low, 2), ",", round(ci_high, 2), "]")

Observaciones CONTROL: 140
Observaciones TREATMENT: 154
GMV promedio semanal CONTROL: 14514.27
GMV promedio semanal TREATMENT: 12331.9
Diferencia absoluta: -2182.37
Lift relativo: -15.04 %
p-value: 0.03757
IC 95% diferencia: [ -4238.41 , -126.33 ]


In [ ]:
# Resumen por grupo para visualización
df_gmv.groupby("variant")["weekly_gmv"].agg(["count", "mean", "std", "min", "max"]).round(2)

,count,mean,std,min,max
variant,,,,,
CONTROL,140,14514.27,9030.90,77.89,35510.84
TREATMENT,154,12331.90,8850.66,77.89,38978.35


### Cómo interpretar los resultados

- Si `p-value < 0.05`, la diferencia se considera estadísticamente significativa al 95%.
- Si el intervalo de confianza 95% **no incluye 0**, eso refuerza que existe una diferencia estadísticamente significativa.
- Si el lift relativo es positivo, `TREATMENT` tuvo mejor desempeño promedio que `CONTROL`.
- Si el p-value es alto o el IC incluye 0, no hay evidencia suficiente para afirmar un efecto claro.

### Pregunta 3 — Resultado en ticket y frecuencia

**Pregunta:**  
¿El efecto viene de tickets más altos, de más transacciones por tienda, o de ambos?

**Objetivo:**  
Descomponer el resultado del experimento para entender el mecanismo detrás del efecto observado en GMV.  
En lugar de ver solo el total vendido, se analizará si `TREATMENT` mejora por:
- mayor cantidad de transacciones por tienda
- mayor ticket promedio
- o una combinación de ambos

In [ ]:
# Ejecutar query y traer la base a pandas
query_b3_ticket_freq = """
WITH experimento AS (
    SELECT DISTINCT
        sp.store_id,
        sp.variant,
        CAST(sp.start_date AS DATE) AS start_date,
        CAST(sp.end_date AS DATE) AS end_date
    FROM store_promotions sp
    WHERE sp.promo_name = 'Exhibicion_Q3_2024'
),
ventas_semanales AS (
    SELECT
        e.store_id,
        e.variant,
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE)) AS week_start,
        SUM(t.total_amount) AS weekly_gmv,
        COUNT(DISTINCT t.transaction_id) AS weekly_transactions,
        AVG(t.total_amount) AS avg_ticket_weekly
    FROM experimento e
    INNER JOIN transactions t
        ON e.store_id = t.store_id
       AND CAST(t.transaction_date AS DATE) BETWEEN e.start_date AND e.end_date
    GROUP BY
        e.store_id,
        e.variant,
        DATE_TRUNC('week', CAST(t.transaction_date AS DATE))
)
SELECT
    store_id,
    variant,
    week_start,
    weekly_gmv,
    weekly_transactions,
    avg_ticket_weekly
FROM ventas_semanales
ORDER BY variant, store_id, week_start;
"""

df_ticket_freq = con.execute(query_b3_ticket_freq).df()
df_ticket_freq.head()

,store_id,variant,week_start,weekly_gmv,weekly_transactions,avg_ticket_weekly
0,TIENDA_001,CONTROL,2024-08-26,3819.98,16,238.748750
1,TIENDA_001,CONTROL,2024-09-02,20770.31,79,262.915316
2,TIENDA_001,CONTROL,2024-09-09,26038.98,98,265.703878
3,TIENDA_001,CONTROL,2024-09-16,23099.66,89,259.546742
4,TIENDA_001,CONTROL,2024-09-23,32731.74,96,340.955625


Esta query arma la base semanal por tienda durante el experimento y calcula:

- **weekly_gmv**: GMV semanal
- **weekly_transactions**: cantidad de transacciones semanales
- **avg_ticket_weekly**: ticket promedio semanal

Con esta base se puede evaluar si `TREATMENT` mejora por volumen (más transacciones), por valor (mejor ticket) o por ambos.

In [ ]:
import numpy as np
from scipy import stats

# Separar grupos
control_tx = df_ticket_freq[df_ticket_freq["variant"] == "CONTROL"]["weekly_transactions"].dropna()
treatment_tx = df_ticket_freq[df_ticket_freq["variant"] == "TREATMENT"]["weekly_transactions"].dropna()

control_ticket = df_ticket_freq[df_ticket_freq["variant"] == "CONTROL"]["avg_ticket_weekly"].dropna()
treatment_ticket = df_ticket_freq[df_ticket_freq["variant"] == "TREATMENT"]["avg_ticket_weekly"].dropna()

# ---------- Función auxiliar para comparación ----------
def comparar_metricas(control, treatment, nombre_metrica):
    mean_control = control.mean()
    mean_treatment = treatment.mean()
    diff_abs = mean_treatment - mean_control
    lift_rel = diff_abs / mean_control if mean_control != 0 else np.nan

    t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)

    n_t = len(treatment)
    n_c = len(control)

    var_t = treatment.var(ddof=1)
    var_c = control.var(ddof=1)

    se_diff = np.sqrt((var_t / n_t) + (var_c / n_c))

    df_welch = ((var_t / n_t + var_c / n_c) ** 2) / (
        ((var_t / n_t) ** 2) / (n_t - 1) + ((var_c / n_c) ** 2) / (n_c - 1)
    )

    t_crit = stats.t.ppf(0.975, df_welch)
    ci_low = diff_abs - t_crit * se_diff
    ci_high = diff_abs + t_crit * se_diff

    return {
        "metric": nombre_metrica,
        "n_control": n_c,
        "n_treatment": n_t,
        "mean_control": mean_control,
        "mean_treatment": mean_treatment,
        "diff_abs": diff_abs,
        "lift_rel_pct": lift_rel * 100,
        "t_stat": t_stat,
        "p_value": p_value,
        "ci95_low": ci_low,
        "ci95_high": ci_high
    }

resultado_tx = comparar_metricas(control_tx, treatment_tx, "weekly_transactions")
resultado_ticket = comparar_metricas(control_ticket, treatment_ticket, "avg_ticket_weekly")

df_resultados_q3 = pd.DataFrame([resultado_tx, resultado_ticket])
df_resultados_q3

,metric,n_control,n_treatment,mean_control,mean_treatment,diff_abs,lift_rel_pct,t_stat,p_value,ci95_low,ci95_high
0,weekly_transactions,140,154,51.871429,44.688312,-7.183117,-13.847926,-1.999494,0.046499,-14.254083,-0.112151
1,avg_ticket_weekly,140,154,284.301415,273.800269,-10.501146,-3.693667,-1.213580,0.225981,-27.538023,6.535730


### Paso estadístico

Ahora se comparan `CONTROL` vs `TREATMENT` en dos métricas:

1. **weekly_transactions** → frecuencia / volumen
2. **avg_ticket_weekly** → ticket promedio

La idea es ver cuál de las dos explica mejor la diferencia observada en GMV.

In [ ]:
# Resumen
for _, row in df_resultados_q3.iterrows():
    print("Métrica:", row["metric"])
    print("  Media CONTROL:", round(row["mean_control"], 2))
    print("  Media TREATMENT:", round(row["mean_treatment"], 2))
    print("  Diferencia absoluta:", round(row["diff_abs"], 2))
    print("  Lift relativo:", round(row["lift_rel_pct"], 2), "%")
    print("  p-value:", round(row["p_value"], 6))
    print("  IC 95%: [", round(row["ci95_low"], 2), ",", round(row["ci95_high"], 2), "]")
    print()

Métrica: weekly_transactions
  Media CONTROL: 51.87
  Media TREATMENT: 44.69
  Diferencia absoluta: -7.18
  Lift relativo: -13.85 %
  p-value: 0.046499
  IC 95%: [ -14.25 , -0.11 ]

Métrica: avg_ticket_weekly
  Media CONTROL: 284.3
  Media TREATMENT: 273.8
  Diferencia absoluta: -10.5
  Lift relativo: -3.69 %
  p-value: 0.225981
  IC 95%: [ -27.54 , 6.54 ]



In [ ]:
# Resumen descriptivo por grupo
df_ticket_freq.groupby("variant")[["weekly_transactions", "avg_ticket_weekly", "weekly_gmv"]].agg(["count", "mean", "std"]).round(2)

weekly_transactions               avg_ticket_weekly               \
                        count   mean    std             count   mean   std   
variant                                                                      
CONTROL                   140  51.87  31.37               140  284.3  81.2   
TREATMENT                 154  44.69  30.09               154  273.8  65.4   

          weekly_gmv                     
               count      mean      std  
variant                                  
CONTROL          140  14514.27  9030.90  
TREATMENT        154  12331.90  8850.66

### Cómo interpretar los resultados

- Si `weekly_transactions` es mayor en `TREATMENT` y el resultado es significativo, entonces el efecto viene de **más frecuencia / más volumen**.
- Si `avg_ticket_weekly` es mayor en `TREATMENT` y el resultado es significativo, entonces el efecto viene de **tickets más altos**.
- Si ambas métricas mejoran, entonces el efecto viene de **ambos componentes**.
- Si una sube y la otra no, eso ayuda a explicar el mecanismo del uplift observado en GMV.

### Pregunta 4 — Decisión de negocio

**Pregunta:**  
¿Implementarías la nueva exhibición en todas las tiendas? Argumenta considerando el p-value, el tamaño del efecto y el costo de implementación. ¿Qué harías si el p-value es 0.08?

**Objetivo:**  
Traducir los resultados del experimento en una decisión de negocio accionable, considerando no solo significancia estadística, sino también tamaño del efecto, costo de implementación y riesgo de escalar una estrategia sin evidencia suficiente.

In [ ]:
decision_resumen = {
    "gmv_mean_control": mean_control,
    "gmv_mean_treatment": mean_treatment,
    "diff_abs": diff_abs,
    "lift_rel_pct": lift_rel * 100,
    "p_value": p_value,
    "ci95_low": ci_low,
    "ci95_high": ci_high
}

decision_resumen

{'gmv_mean_control': np.float64(14514.268428571428),
 'gmv_mean_treatment': np.float64(12331.900324675324),
 'diff_abs': np.float64(-2182.368103896104),
 'lift_rel_pct': np.float64(-15.036018622889038),
 'p_value': np.float64(0.037570010817862834),
 'ci95_low': np.float64(-4238.406901770106),
 'ci95_high': np.float64(-126.32930602210263)}

In [ ]:
# Regla simple de apoyo para interpretación
if p_value < 0.05 and diff_abs > 0:
    recomendacion = "ESCALAR"
elif p_value >= 0.05 and p_value <= 0.10 and diff_abs > 0:
    recomendacion = "PILOTO_EXTENDIDO"
else:
    recomendacion = "NO_ESCALAR_AUN"

print("Recomendación sugerida:", recomendacion)
print("GMV promedio CONTROL:", round(mean_control, 2))
print("GMV promedio TREATMENT:", round(mean_treatment, 2))
print("Diferencia absoluta:", round(diff_abs, 2))
print("Lift relativo:", round(lift_rel * 100, 2), "%")
print("p-value:", round(p_value, 6))
print("IC 95%:", "[", round(ci_low, 2), ",", round(ci_high, 2), "]")

Recomendación sugerida: NO_ESCALAR_AUN
GMV promedio CONTROL: 14514.27
GMV promedio TREATMENT: 12331.9
Diferencia absoluta: -2182.37
Lift relativo: -15.04 %
p-value: 0.03757
IC 95%: [ -4238.41 , -126.33 ]


### Cómo interpretar esta decisión

Para decidir si conviene implementar la nueva exhibición en todas las tiendas, no basta con ver si `TREATMENT` vendió más. Se deben combinar tres factores:

1. **Significancia estadística**
   - Si `p-value < 0.05`, existe evidencia más sólida de que el efecto no se debe al azar.
   - Si `p-value` está cerca de 0.05 pero no lo cruza (por ejemplo, **0.08**), la señal existe pero es insuficiente para una afirmación fuerte.

2. **Tamaño del efecto**
   - Aunque el resultado sea significativo, el lift debe ser lo suficientemente grande para justificar el cambio.
   - Un efecto muy pequeño puede no compensar el esfuerzo o costo operativo.

3. **Costo de implementación**
   - Si la exhibición es barata y fácil de escalar, se puede aceptar un umbral de evidencia algo más flexible.
   - Si implica costo alto, uso de espacio o complejidad operativa, la evidencia debe ser más fuerte antes de escalar.